# BanksOnSale Data: one bank, eight quarters

A reproducible financial panel for a class, notebook or application. Original source: FDIC BankFind Suite. This example uses a **frozen 2026Q2 sample**, not a latest-data claim. After installing the package, no source network requests are needed. Use a fresh Python 3.10+ notebook environment.


In [ ]:
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("banksonsale_data") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install",
        "https://github.com/anshuman-pandey/banksonsale-data/releases/download/v0.1.0/banksonsale_data-0.1.0-py3-none-any.whl"])


In [ ]:
from banksonsale_data import load_sample

panel = load_sample(58665)
print(panel.document["bank"]["name"])
print(panel.document["requested_dates"])
print(panel.document["coverage"])


## Inspect explicit units

Money is USD. Ratios are percent: 12.5 means 12.5%. Missing stays `None`; zero stays zero. CET1 uses RBCT1CER; leverage uses RBC1AAJ. The bank is distinct from its listed parent.


In [ ]:
latest = panel.rows[-1]
for field in ["report_date", "assets_usd", "deposits_usd", "cet1_ratio_pct", "leverage_ratio_pct", "roa_pct"]:
    print(field, latest[field])
print(panel.document["metric_definitions"]["assets_usd"])
assert latest["report_date"] == "2026-06-30"


## Keep the evidence with the export

The CSV sidecar records the source URLs, retrieval times, raw source responses, metric definitions and file hash. The JSON snapshot can be loaded and checked later. Hashes detect content changes, not whether the original filing is economically correct.


In [ ]:
from pathlib import Path
from banksonsale_data import BankPanel

output = Path("outputs/banksonsale-example")
panel.export(output / "live-oak.csv")
panel.export(output / "live-oak.json")
restored = BankPanel.load(output / "live-oak.json")
assert restored.rows == panel.rows
print(panel.dataset_id)
print(output / "live-oak.manifest.json")


## Optional live request

Use `bank(58665).financials(quarters=8, through="2026Q2")` to fetch a new source snapshot for an explicit period. Omitting `through` selects that bank's latest available report date, which can differ from another bank's latest date.

For hosted saved research and monitoring, [continue in BanksOnSale](https://www.banksonsale.com/banks/58665). Hosted data editions are displayed separately; coverage can differ. The open package works without registering. Cite **BanksOnSale Data, sourced from FDIC BankFind Suite**, with the report dates and dataset ID.
